In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import timm, types, sys, h5py, wandb, random, copy
from pathlib import Path
from tqdm.auto import tqdm
from scipy.stats import spearmanr
from torch.utils.data import DataLoader, WeightedRandomSampler, TensorDataset
from peft import LoraConfig
from peft.tuners.lora import LoraModel
import torchvision.transforms as T

sys.path.append(".")
from src.dataset import HistologicalImageDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [2]:
DATA_DIR     = "/data/BREAKHIS"
DATASET_NAME = Path(DATA_DIR).name
BASE_CKPT    = Path("checkpoints")
BASE_CACHE   = Path("/data/data_cache")


# Target intermedi con pesi decrescenti.
# Ispirazione: paper mostra che layer profondi sono più
# importanti → peso maggiore al layer finale.
# Layer vicini = supervisione ausiliaria a basso peso.
MULTI_TARGET_WEIGHTS = {
    23: 1.0,   # target principale
    20: 0.3,   # supervisione ausiliaria
    15: 0.15,  # supervisione ausiliaria
}

CFG = dict(
    data_dir        = DATA_DIR,
    dataset_name    = DATASET_NAME,
    img_size        = 224,
    batch_size      = 64,
    num_workers     = 8,
    seed            = 42,
    output_dir      = BASE_CKPT / DATASET_NAME / "uni_finetuned",
    dataset_cache   = BASE_CACHE / f"{DATASET_NAME}_attn_distill_dataset.h5",
    forecaster_dir  = BASE_CKPT / DATASET_NAME / "forecaster_distill",

    layer_source    = 2,           # embedding di input al forecaster
    layer_target    = 23,          # target principale
    multi_targets   = MULTI_TARGET_WEIGHTS,

    # Architettura
    hidden          = 256,
    n_heads         = 4,
    n_layers        = 2,
    dropout         = 0.2,

    # Training — da Li et al. Table 16
    epochs          = 30,
    lr              = 1e-4,        # blr=1e-4 dal paper
    weight_decay    = 0.3,         # 0.3 dal paper (non 0.05)
    beta2           = 0.95,        # β2=0.95 dal paper
    atd_weight      = 3.0,         # λ=3 dal paper Appendix B.5
    ema_decay       = 0.9999,      # EMA dei pesi
    warmup_epochs   = 5,           # warmup

    wandb_project   = "attention-distillation",
)

for k in ["output_dir", "dataset_cache", "forecaster_dir"]:
    CFG[k] = Path(CFG[k])
CFG["dataset_cache"].parent.mkdir(parents=True, exist_ok=True)
CFG["output_dir"].mkdir(parents=True, exist_ok=True)
CFG["forecaster_dir"].mkdir(parents=True, exist_ok=True)

torch.manual_seed(CFG["seed"])
torch.cuda.manual_seed_all(CFG["seed"])
np.random.seed(CFG["seed"])
random.seed(CFG["seed"])
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
print(f"Dataset: {DATASET_NAME}")
print(f"Multi-target layers: {list(CFG['multi_targets'].keys())}")


Dataset: BREAKHIS
Multi-target layers: [23, 20, 15]


In [3]:
train_tf = T.Compose([
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomApply([T.RandomRotation((90,90))], p=0.5),
    T.RandomApply([T.ColorJitter(0.2,0.2,0.1,0.05)], p=0.5),
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])
eval_tf = T.Compose([
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])

train_ds = HistologicalImageDataset(f"{CFG['data_dir']}/train", transform=train_tf)
val_ds   = HistologicalImageDataset(f"{CFG['data_dir']}/val",   transform=eval_tf)
test_ds  = HistologicalImageDataset(f"{CFG['data_dir']}/test",  transform=eval_tf)

counts  = np.bincount(train_ds.labels)
weights = torch.from_numpy((1.0/counts)[train_ds.labels]).double()
sampler = WeightedRandomSampler(weights, len(weights), replacement=True)
kw = dict(batch_size=CFG["batch_size"], num_workers=CFG["num_workers"],
          pin_memory=True, persistent_workers=True)
train_loader = DataLoader(train_ds, sampler=sampler, drop_last=False, **kw)
val_loader   = DataLoader(val_ds,   shuffle=False, **kw)
test_loader  = DataLoader(test_ds,  shuffle=False, **kw)
CLASS_NAMES  = train_ds.class_names
N_CLASSES    = len(CLASS_NAMES)
print(f"Train={len(train_ds)} Val={len(val_ds)} Test={len(test_ds)}")


Loading from /data/BREAKHIS/train...


Loading dataset from disk:   0%|          | 0/29 [00:00<?, ?it/s]

Loaded 25880 samples, 8 classes
Class distribution:
  adenosis: 1139 (4.4%)
  fibroadenoma: 3685 (14.2%)
  phyllodes_tumor: 1419 (5.5%)
  tubular_adenoma: 1642 (6.3%)
  ductal_carcinoma: 11717 (45.3%)
  lobular_carcinoma: 1927 (7.4%)
  mucinous_carcinoma: 2446 (9.5%)
  papillary_carcinoma: 1905 (7.4%)
Loading from /data/BREAKHIS/val...
Loaded 6832 samples, 8 classes
Class distribution:
  adenosis: 541 (7.9%)
  fibroadenoma: 692 (10.1%)
  phyllodes_tumor: 423 (6.2%)
  tubular_adenoma: 601 (8.8%)
  ductal_carcinoma: 2769 (40.5%)
  lobular_carcinoma: 601 (8.8%)
  mucinous_carcinoma: 757 (11.1%)
  papillary_carcinoma: 448 (6.6%)
Loading from /data/BREAKHIS/test...
Loaded 6833 samples, 8 classes
Class distribution:
  adenosis: 540 (7.9%)
  fibroadenoma: 693 (10.1%)
  phyllodes_tumor: 423 (6.2%)
  tubular_adenoma: 602 (8.8%)
  ductal_carcinoma: 2769 (40.5%)
  lobular_carcinoma: 602 (8.8%)
  mucinous_carcinoma: 757 (11.1%)
  papillary_carcinoma: 447 (6.5%)
Train=25880 Val=6832 Test=6833


In [4]:
class UNILoRAClassifier(nn.Module):
    def __init__(self, n_classes, dropout=0.1):
        super().__init__()
        backbone = timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True,
                                      init_values=1e-5, dynamic_img_size=True)
        lora_config = LoraConfig(r=8, lora_alpha=32,
            target_modules=["qkv","proj","fc1","fc2"], lora_dropout=0.1, bias="none")
        self.backbone = LoraModel(backbone, lora_config, adapter_name="default")
        self.head = nn.Sequential(nn.LayerNorm(1024), nn.Dropout(dropout),
                                   nn.Linear(1024, n_classes))
    def forward(self, x):
        return self.head(self.backbone(x))

model = UNILoRAClassifier(N_CLASSES).to(device)
ckpt  = torch.load(CFG["output_dir"] / "best_model.pt", map_location=device)
model.load_state_dict(ckpt, strict=False)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print("Classificatore caricato e frozen")

Classificatore caricato e frozen


In [5]:
all_target_layers = sorted(CFG["multi_targets"].keys())

def collect_and_save_dataset(model, loaders_dict, device,
                              layer_source, all_target_layers, save_path):
    with h5py.File(save_path, 'w') as f:
        for split_name, loader in loaders_dict.items():
            print(f"\nRaccolta {split_name}...")
            n_total   = len(loader.dataset)
            n_patches = 196
            embed_dim = 1024

            grp      = f.create_group(split_name)
            ds_label = grp.create_dataset("labels",
                shape=(n_total,), maxshape=(None,), dtype='i4', compression="gzip")
            ds_emb   = grp.create_dataset(f"emb_layer{layer_source}",
                shape=(n_total, n_patches, embed_dim),
                maxshape=(None, n_patches, embed_dim),
                dtype='f2', compression="gzip", chunks=(64, n_patches, embed_dim))
            ds_attns = {t: grp.create_dataset(f"attn_layer{t}",
                shape=(n_total, n_patches), maxshape=(None, n_patches),
                dtype='f2', compression="gzip", chunks=(64, n_patches))
                for t in all_target_layers}

            orig  = {}
            cache = {}

            def make_hook(idx):
                def fwd(self, x):
                    B, N, C = x.shape
                    qkv  = self.qkv(x).reshape(B,N,3,self.num_heads,
                                     self.head_dim).permute(2,0,3,1,4)
                    q, k, v = qkv.unbind(0)
                    q, k   = self.q_norm(q), self.k_norm(k)
                    attn   = (q @ k.transpose(-2,-1) * self.scale).softmax(-1)
                    if idx == layer_source:
                        cache[f"emb_{idx}"] = x[:,1:].detach().cpu().half()
                    if idx in all_target_layers:
                        cache[f"attn_{idx}"] = attn[:,:,0,1:].mean(1).detach().cpu().half()
                    x = (self.attn_drop(attn) @ v).transpose(1,2).reshape(B,N,C)
                    return self.proj_drop(self.proj(x))
                return fwd

            for i, block in enumerate(model.backbone.model.blocks):
                orig[i] = block.attn.forward
                block.attn.forward = types.MethodType(make_hook(i), block.attn)

            FLUSH_EVERY = 16
            buf_labels  = []
            buf_emb     = []
            buf_attns   = {t: [] for t in all_target_layers}

            def flush(ptr):
                if not buf_labels: return ptr
                B = sum(len(x) for x in buf_labels)
                ds_label[ptr:ptr+B] = np.concatenate(buf_labels)
                ds_emb[ptr:ptr+B]   = torch.cat(buf_emb).numpy()
                for t in all_target_layers:
                    ds_attns[t][ptr:ptr+B] = torch.cat(buf_attns[t]).numpy()
                buf_labels.clear(); buf_emb.clear()
                for t in all_target_layers: buf_attns[t].clear()
                return ptr + B

            ptr = 0
            with torch.no_grad():
                for i_batch, (imgs, labels) in enumerate(tqdm(loader, desc=split_name)):
                    cache.clear()
                    model(imgs.to(device))
                    buf_labels.append(labels.numpy())
                    buf_emb.append(cache[f"emb_{layer_source}"])
                    for t in all_target_layers:
                        buf_attns[t].append(cache[f"attn_{t}"])
                    if (i_batch + 1) % FLUSH_EVERY == 0:
                        ptr = flush(ptr)

            ptr = flush(ptr)
            for i, block in enumerate(model.backbone.model.blocks):
                block.attn.forward = orig[i]
            print(f"  {split_name}: {ptr} sample salvati")

if not CFG["dataset_cache"].exists():
    collect_and_save_dataset(model,
        {"train": train_loader, "val": val_loader, "test": test_loader},
        device, CFG["layer_source"], all_target_layers, CFG["dataset_cache"])
else:
    print(f"Cache trovata: {CFG['dataset_cache']}")
    with h5py.File(CFG["dataset_cache"], 'r') as f:
        for split in f.keys():
            print(f"  {split}: {len(f[split]['labels'])} sample")
            missing = [t for t in all_target_layers
                       if f"attn_layer{t}" not in f[split]]
            if missing:
                print(f"  ⚠️  Layer mancanti: {missing} — cancella il cache e riesegui")


Cache trovata: /data/data_cache/BREAKHIS_attn_distill_dataset.h5
  test: 6833 sample
  train: 25880 sample
  val: 6832 sample


In [6]:
class H5MultiTargetDataset(torch.utils.data.Dataset):
    """
    Restituisce (emb, {layer: attn}, label) per ogni sample.
    Tutti i target vengono caricati in un unico accesso H5.
    """
    def __init__(self, h5_path, split, layer_source, target_layers):
        self.h5_path       = str(h5_path)
        self.split         = split
        self.layer_source  = layer_source
        self.target_layers = target_layers
        self._file         = None
        with h5py.File(h5_path, 'r') as f:
            self.length = len(f[split]["labels"])

    def _get_file(self):
        if self._file is None:
            self._file = h5py.File(self.h5_path, 'r')
        return self._file

    def __len__(self): return self.length

    def __getitem__(self, idx):
        grp = self._get_file()[self.split]
        emb     = torch.from_numpy(grp[f"emb_layer{self.layer_source}"][idx]).float()
        targets = {t: torch.from_numpy(grp[f"attn_layer{t}"][idx]).float()
                   for t in self.target_layers}
        label   = int(grp["labels"][idx])
        return emb, targets, label


def collate_multi_target(batch):
    embs   = torch.stack([b[0] for b in batch])
    labels = torch.tensor([b[2] for b in batch])
    target_layers = list(batch[0][1].keys())
    targets = {t: torch.stack([b[1][t] for b in batch]) for t in target_layers}
    return embs, targets, labels


def make_loaders(h5_path, layer_source, target_layers):
    ds_kw = dict(layer_source=layer_source, target_layers=target_layers)
    ld_kw = dict(batch_size=64, num_workers=4, pin_memory=True,
                 persistent_workers=False, collate_fn=collate_multi_target)
    return (
        DataLoader(H5MultiTargetDataset(h5_path, "train", **ds_kw), shuffle=True,  **ld_kw),
        DataLoader(H5MultiTargetDataset(h5_path, "val",   **ds_kw), shuffle=False, **ld_kw),
        DataLoader(H5MultiTargetDataset(h5_path, "test",  **ds_kw), shuffle=False, **ld_kw),
    )

In [7]:
class AttentionForecaster(nn.Module):
    def __init__(self, embed_dim=1024, hidden=256, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.input_proj  = nn.Linear(embed_dim, hidden)
        self.cls_query   = nn.Parameter(torch.randn(1,1,hidden)*0.02)
        self.self_attn   = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=hidden, nhead=n_heads,
                dim_feedforward=hidden*2, dropout=dropout,
                batch_first=True, norm_first=True) for _ in range(n_layers)])
        self.cross_attn  = nn.ModuleList([
            nn.MultiheadAttention(hidden, n_heads, dropout=dropout, batch_first=True)
            for _ in range(n_layers)])
        self.cross_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.norm        = nn.LayerNorm(hidden)
        self.score_head  = nn.Sequential(
            nn.Linear(hidden*2,128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128,1))

    def forward(self, x):
        B, N, D = x.shape
        x = self.input_proj(x)
        for sa in self.self_attn: x = sa(x)
        cls = self.cls_query.expand(B,-1,-1)
        for ca, norm in zip(self.cross_attn, self.cross_norms):
            cls_out, _ = ca(cls, x, x); cls = norm(cls + cls_out)
        x_n = self.norm(x)
        return self.score_head(torch.cat([x_n, cls.expand(-1,N,-1)], dim=-1)).squeeze(-1).softmax(-1)


In [8]:
def attention_distill_loss(pred, targets_dict, weights_dict, atd_weight):
    """
    pred:         [B, 196] softmax output del forecaster
    targets_dict: {layer: [B, 196]} attenzione teacher
    weights_dict: {layer: float} pesi per ciascun target
    atd_weight:   λ globale (=3 dal paper)
    """
    loss = 0.
    for layer, target in targets_dict.items():
        w    = weights_dict[layer]
        # Cross-entropy: -sum(target * log(pred + eps))
        ce   = -(target * (pred + 1e-8).log()).sum(-1).mean()
        loss += w * ce
    return atd_weight * loss

In [9]:
class EMA:
    def __init__(self, model, decay=0.9999):
        self.decay   = decay
        self.shadow  = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for s_p, m_p in zip(self.shadow.parameters(), model.parameters()):
            s_p.data.mul_(self.decay).add_(m_p.data, alpha=1 - self.decay)

    def get_model(self): return self.shadow


In [10]:
def train_forecaster(cfg, device):
    run_name = f"distill_src{cfg['layer_source']:02d}_tgt{cfg['layer_target']:02d}"
    print(f"\n{'='*60}\n  {run_name}\n{'='*60}")

    wandb.init(
        project = cfg["wandb_project"],
        name    = run_name,
        config  = {k: str(v) if isinstance(v, Path) else v
                   for k, v in cfg.items()},
        tags    = [f"src{cfg['layer_source']}", "ce-loss", "ema", DATASET_NAME],
        reinit  = True,
    )

    tr_ld, vl_ld, ts_ld = make_loaders(
        cfg["dataset_cache"], cfg["layer_source"],
        list(cfg["multi_targets"].keys()))

    forecaster = AttentionForecaster(
        embed_dim=1024, hidden=cfg["hidden"],
        n_heads=cfg["n_heads"], n_layers=cfg["n_layers"],
        dropout=cfg["dropout"]).to(device)

    ema = EMA(forecaster, decay=cfg["ema_decay"])

    # Optimizer — β2=0.95 come da paper Table 16
    opt = torch.optim.AdamW(forecaster.parameters(),
                             lr=cfg["lr"],
                             weight_decay=cfg["weight_decay"],
                             betas=(0.9, cfg["beta2"]))

    # Warmup + cosine annealing
    def lr_lambda(step):
        warmup = cfg["warmup_epochs"] * len(tr_ld)
        total  = cfg["epochs"] * len(tr_ld)
        if step < warmup:
            return step / max(1, warmup)
        progress = (step - warmup) / max(1, total - warmup)
        return 0.5 * (1 + np.cos(np.pi * progress))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    save_path    = cfg["forecaster_dir"] / f"best_{run_name}.pt"
    best_val_ce  = float('inf')
    best_val_rho = -1.
    step         = 0
    main_target  = cfg["layer_target"]

    for epoch in range(cfg["epochs"]):

        # ── Train ──
        forecaster.train()
        total_ce = 0.

        for emb, targets, _ in tqdm(tr_ld, leave=False, desc=f"Ep{epoch+1}"):
            emb = emb.to(device)
            targets_dev = {t: v.to(device) for t, v in targets.items()}

            pred = forecaster(emb)
            loss = attention_distill_loss(
                pred, targets_dev, cfg["multi_targets"], cfg["atd_weight"])

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(forecaster.parameters(), 1.0)
            opt.step(); sched.step()
            ema.update(forecaster)
            total_ce += loss.item()
            step += 1

        train_ce = total_ce / len(tr_ld)

        # ── Val con EMA ──
        ema_model = ema.get_model()
        val_ce, rho_list = 0., []

        with torch.no_grad():
            for emb, targets, _ in vl_ld:
                emb = emb.to(device)
                targets_dev = {t: v.to(device) for t, v in targets.items()}
                pred = ema_model(emb)
                val_ce += attention_distill_loss(
                    pred, targets_dev, cfg["multi_targets"], cfg["atd_weight"]).item()
                # Spearman sul target principale
                main_tgt = targets[main_target]
                for b in range(len(emb)):
                    rho, _ = spearmanr(pred[b].cpu().numpy(),
                                       main_tgt[b].numpy())
                    rho_list.append(rho)

        val_ce  /= len(vl_ld)
        val_rho  = np.nanmean(rho_list)
        lr_now   = opt.param_groups[0]["lr"]

        wandb.log({
            "epoch"    : epoch+1,
            "train/ce" : train_ce,
            "val/ce"   : val_ce,
            "val/rho"  : val_rho,
            "lr"       : lr_now,
        })

        if val_ce < best_val_ce:
            best_val_ce  = val_ce
            best_val_rho = val_rho
            torch.save(ema_model.state_dict(), save_path)

        if (epoch+1) % 5 == 0:
            print(f"  Ep{epoch+1:02d} | ce={val_ce:.4f} ρ={val_rho:.3f} "
                  f"best_ρ={best_val_rho:.3f} lr={lr_now:.2e}")

    # ── Test ──
    ema_model.load_state_dict(torch.load(save_path))
    ema_model.eval()
    test_rho_f, test_rho_n = [], []

    with h5py.File(cfg["dataset_cache"], 'r') as f_h5:
        emb_all  = torch.from_numpy(
            f_h5["test"][f"emb_layer{cfg['layer_source']}"][:]).float()
        tgt_all  = torch.from_numpy(
            f_h5["test"][f"attn_layer{main_target}"][:]).float()

    with torch.no_grad():
        for emb, tgt in DataLoader(TensorDataset(emb_all, tgt_all),
                                    batch_size=64, shuffle=False):
            pred = ema_model(emb.to(device)).cpu()
            for b in range(len(emb)):
                t = tgt[b].numpy()
                rho_f, _ = spearmanr(pred[b].numpy(), t)
                rho_n, _ = spearmanr(emb[b].norm(dim=-1).numpy(), t)
                test_rho_f.append(rho_f); test_rho_n.append(rho_n)

    test_rho_f = np.nanmean(test_rho_f)
    test_rho_n = np.nanmean(test_rho_n)

    wandb.log({"test/rho_forecaster": test_rho_f,
               "test/rho_token_norm": test_rho_n,
               "test/delta"         : test_rho_f - test_rho_n,
               "best_val_rho"       : best_val_rho})
    print(f"\n  Test ρ forecaster (EMA): {test_rho_f:.3f}")
    print(f"  Test ρ token norm:       {test_rho_n:.3f}")
    print(f"  Δ:                       {test_rho_f - test_rho_n:+.3f}")
    wandb.finish()

    return {"layer_source"       : cfg["layer_source"],
            "test_rho_forecaster": test_rho_f,
            "test_rho_token_norm": test_rho_n,
            "best_val_rho"       : best_val_rho,
            "best_val_ce"        : best_val_ce}

# %%
result = train_forecaster(CFG, device)
print(result)



  distill_src02_tgt23


wandb: Currently logged in as: vincenzo-civale (vincenzo-civale-universi-degli-studi-di-firenze) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Ep1:   0%|          | 0/405 [00:00<?, ?it/s]

Ep2:   0%|          | 0/405 [00:00<?, ?it/s]

Ep3:   0%|          | 0/405 [00:00<?, ?it/s]

...

Ep29:   0%|          | 0/405 [00:00<?, ?it/s]

Ep30:   0%|          | 0/405 [00:00<?, ?it/s]

  Ep30 | ce=16.8841 ρ=0.640 best_ρ=0.640 lr=0.00e+00

  Test ρ forecaster (EMA): 0.639
  Test ρ token norm:       0.111
  Δ:                       +0.527


best_val_rho,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,▂▄▅▇█████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
test/delta,▁
test/rho_forecaster,▁
test/rho_token_norm,▁
train/ce,█▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/ce,██▇▇▇▆▆▆▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
val/rho,▁▂▃▃▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇█████████
best_val_rho,0.63987
epoch,30


{'layer_source': 2, 'test_rho_forecaster': np.float64(0.6385879945852594), 'test_rho_token_norm': np.float64(0.11112838784297117), 'best_val_rho': np.float64(0.6398705865021395), 'best_val_ce': 16.884090601840867}
